# Stratified Group K-Fold validation


**Important:** Place `merged_physics.csv` and `uav_strict_utils.py` in the same folder as this notebook before running.  
The notebook saves `metrics.csv`, `classification_report.txt`, `predictions.csv`, `confusion_matrix.pdf`, `metrics_bar.pdf`, `training_curve.pdf`, and the trained `.keras` model for each technique.


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from uav_strict_utils import *

INPUT_FILE = 'merged_physics.csv'
RESULT_ROOT = Path('Stratified_Group_KFold_Results')
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 50
STRIDE = 50
EPOCHS = 80
BATCH_SIZE = 16

_, X, y, y_text, groups, label_encoder, feature_cols = prepare_windows(
    INPUT_FILE, window_size=WINDOW_SIZE, stride=STRIDE, filter_active=True
)

n_splits = min(3, len(np.unique(groups)))
cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
all_fold_metrics = []

for fold, (trainval_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
    print('\n' + '#'*80)
    print(f'Fold {fold}/{n_splits}')
    print('#'*80)

    y_trainval = y[trainval_idx]
    train_idx, val_idx = train_test_split(
        trainval_idx, test_size=0.20, stratify=y_trainval, random_state=SEED + fold
    )

    X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
    y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
    X_train, X_val, X_test, scaler = scale_by_train(X_train, X_val, X_test)

    print('Test files:', np.unique(groups[test_idx]))
    print('Train:', X_train.shape, pd.Series(label_encoder.inverse_transform(y_train)).value_counts().to_dict())
    print('Validation:', X_val.shape, pd.Series(label_encoder.inverse_transform(y_val)).value_counts().to_dict())
    print('Test:', X_test.shape, pd.Series(label_encoder.inverse_transform(y_test)).value_counts().to_dict())

    for model_name in MODEL_NAMES:
        metrics = train_one_model(
            model_name, X_train, y_train, X_val, y_val, X_test, y_test,
            label_encoder, feature_cols, RESULT_ROOT / f'Fold_{fold}' / model_name,
            epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle_labels=False,
            fold_name=f'Fold_{fold}'
        )
        metrics['Model'] = model_name
        metrics['Fold'] = fold
        all_fold_metrics.append(metrics)

fold_df = pd.DataFrame(all_fold_metrics)
fold_df.to_csv(RESULT_ROOT / 'fold_level_metrics.csv', index=False)
summary_df = fold_df.groupby('Model').agg(['mean', 'std'])
summary_df.to_csv(RESULT_ROOT / 'cross_validation_mean_std.csv')
print(summary_df)
summary_df


C:\Users\Hp\miniconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Idle rows removed: 47
Dataset shape after cleaning/filtering: (1396, 33)
Number of numerical features: 31
Class counts after filtering:
class_label
bend      591
normal    589
crack     216
Name: count, dtype: int64

Created non-overlapping windows
Total windows: 21
Window class counts:
bend      9
normal    9
crack     3
Name: count, dtype: int64
Source files: 9

################################################################################
Fold 1/3
################################################################################
Test files: ['Propeller_Data_1045_bend1.csv' 'Propeller_Data_1045_bend3.csv'
 'Propeller_Data_1045_crack3.csv']
Train: (11, 50, 31) {'normal': 7, 'bend': 2, 'crack': 2}
Validation: (3, 50, 31) {'normal': 2, 'bend': 1}
Test: (7, 50, 31) {'bend': 6, 'crack': 1}

Epoch 1/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.2727 - loss: 1.3220 - val_accuracy: 0.0000e+00 - val_loss: 1.0693 - learning_rate: 0.0010
Epoch 2/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step 

Accuracy           Precision              Recall  \
                            mean       std      mean       std      mean   
Model                                                                      
1D_CNN                  0.761905  0.412393  0.814815  0.320750  0.796296   
CNN_LSTM                0.571429  0.377964  0.592593  0.357172  0.518519   
LSTM                    0.714286  0.285714  0.722222  0.254588  0.666667   
Multimodal_Transformer  0.666667  0.459221  0.481481  0.462592  0.555556   
TCN                     0.380952  0.436436  0.305556  0.292657  0.259259   
Transformer             0.666667  0.459221  0.555556  0.384900  0.555556   

                                   F1Score           Precision_weighted  \
                             std      mean       std               mean   
Model                                                                     
1D_CNN                  0.352825  0.753968  0.426139           0.968254   
CNN_LSTM                0.449050  0.500000  0.440959           0.920635   
LSTM                    0.293972  0.662963  0.310383           0.976190   
Multimodal_Transformer  0.384900  0.500000  0.440959           0.634921   
TCN                     0.357172  0.261905  0.320324           0.535714   
Transformer             0.384900  0.555556  0.384900           0.666667   

                                 Recall_weighted           F1Score_weighted  \
                             std            mean       std             mean   
Model                                                                         
1D_CNN                  0.054986        0.761905  0.412393         0.772109   
CNN_LSTM                0.072739        0.571429  0.377964         0.642857   
LSTM                    0.041239        0.714286  0.285714         0.784127   
Multimodal_Transformer  0.513609        0.666667  0.459221         0.642857   
TCN                     0.467025        0.380952  0.436436         0.408163   
Transformer             0.459221        0.666667  0.459221         0.666667   

                                 Fold       
                             std mean  std  
Model                                       
1D_CNN                  0.394719  2.0  1.0  
CNN_LSTM                0.311350  2.0  1.0  
LSTM                    0.241186  2.0  1.0  
Multimodal_Transformer  0.500000  2.0  1.0  
TCN                     0.398351  2.0  1.0  
Transformer             0.459221  2.0  1.0